# Home task: decision trees

### Load dataset and prepare data


In [105]:
# load dataset
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

cancer = load_breast_cancer()

# take data and labels
X = cancer.data
y = cancer.target
labels = cancer.target_names
features = cancer.feature_names

print('labels:', labels)
print('features:', features)

# split data into train and test
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=0)

labels: ['malignant' 'benign']
features: ['mean radius' 'mean texture' 'mean perimeter' 'mean area'
 'mean smoothness' 'mean compactness' 'mean concavity'
 'mean concave points' 'mean symmetry' 'mean fractal dimension'
 'radius error' 'texture error' 'perimeter error' 'area error'
 'smoothness error' 'compactness error' 'concavity error'
 'concave points error' 'symmetry error' 'fractal dimension error'
 'worst radius' 'worst texture' 'worst perimeter' 'worst area'
 'worst smoothness' 'worst compactness' 'worst concavity'
 'worst concave points' 'worst symmetry' 'worst fractal dimension']


### Decision Tree model


In [106]:
# import model
from sklearn.tree import DecisionTreeClassifier
from sklearn import tree
import graphviz

# create and train model
clf = DecisionTreeClassifier(max_depth=3, random_state=0)
clf.fit(X_train, y_train)

# check accuracy
print("train accuracy= {:.3%}".format(clf.score(X_train, y_train)))
print("test accuracy= {:.3%}".format(clf.score(X_test, y_test)))

# visualize tree
graph_viz = tree.export_graphviz(
    clf,
    out_file=None,
    feature_names=features,
    class_names=labels,
    filled=True
)

graph = graphviz.Source(graph_viz)
graph.view(cleanup=True)

train accuracy= 97.653%
test accuracy= 93.706%


'Source.gv.pdf'

### Random Forest model


In [107]:
# import model
from sklearn.ensemble import RandomForestClassifier

# create and train model
clf = RandomForestClassifier(random_state=0)
clf.fit(X_train, y_train)

# check accuracy
print("train accuracy= {:.3%}".format(clf.score(X_train, y_train)))
print("test accuracy= {:.3%}".format(clf.score(X_test, y_test)))

train accuracy= 100.000%
test accuracy= 97.203%


### Gradient Boosting model


In [108]:
# import model
from sklearn.ensemble import GradientBoostingClassifier

# create and train model
clf = GradientBoostingClassifier(random_state=0)
clf.fit(X_train, y_train)

# check accuracy
print("train accuracy= {:.3%}".format(clf.score(X_train, y_train)))
print("test accuracy= {:.3%}".format(clf.score(X_test, y_test)))

train accuracy= 100.000%
test accuracy= 96.503%


### XGBoost model


In [109]:
# import model
from xgboost import XGBClassifier

# create and train model
clf = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
clf.fit(X_train, y_train)

# check accuracy
print("train accuracy= {:.3%}".format(clf.score(X_train, y_train)))
print("test accuracy= {:.3%}".format(clf.score(X_test, y_test)))

train accuracy= 100.000%
test accuracy= 98.601%


## Additional tasks

## 1) Binary Classification



### Binary Classification: Income Prediction
In this task, I predict whether a person earns more than 50k per year.
I use a census dataset and an XGBoost classifier.

In [110]:
# load dataset
from sklearn.datasets import fetch_openml
import pandas as pd

data_class = fetch_openml(name='adult', version=2, as_frame=True)
df_income = data_class.frame

# show first rows
df_income.head()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,class
0,25.0,Private,226802.0,11th,7.0,Never-married,Machine-op-inspct,Own-child,Black,Male,0.0,0.0,40.0,United-States,<=50K
1,38.0,Private,89814.0,HS-grad,9.0,Married-civ-spouse,Farming-fishing,Husband,White,Male,0.0,0.0,50.0,United-States,<=50K
2,28.0,Local-gov,336951.0,Assoc-acdm,12.0,Married-civ-spouse,Protective-serv,Husband,White,Male,0.0,0.0,40.0,United-States,>50K
3,44.0,Private,160323.0,Some-college,10.0,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688.0,0.0,40.0,United-States,>50K
4,18.0,NaN,103497.0,Some-college,10.0,Never-married,NaN,Own-child,White,Female,0.0,0.0,30.0,United-States,<=50K


### Data preprocessing

In [111]:
# separate features and target
X = df_income.drop('class', axis=1)
y = df_income['class']

# fill missing values correctly
for col in X.columns:
    if str(X[col].dtype) == 'category' or X[col].dtype == 'object':
        X[col] = X[col].astype('object').fillna('missing')
    else:
        X[col] = X[col].fillna(X[col].median())

# encode categorical features
X = pd.get_dummies(X)

# encode target
y = (y == '>50K').astype(int)

### Train XGBoost Classifier

In [112]:
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

# split data
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=0)

# create model
clf = XGBClassifier(use_label_encoder=False, eval_metric='logloss')

# train model
clf.fit(X_train, y_train)

# evaluate
print("train accuracy= {:.3%}".format(clf.score(X_train, y_train)))
print("test accuracy= {:.3%}".format(clf.score(X_test, y_test)))

train accuracy= 89.667%
test accuracy= 87.315%


## 2) Regression

### Regression: Forecasting House Prices

In this task, I predict house prices using the XGBoost regressor.

In [113]:
# load dataset
data_reg = fetch_openml(name='house_prices', version=1, as_frame=True)
df_house = data_reg.frame

df_house.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1.0,60.0,RL,65.0,8450.0,Pave,None,Reg,Lvl,AllPub,...,0.0,None,None,None,0.0,2.0,2008.0,WD,Normal,208500.0
1,2.0,20.0,RL,80.0,9600.0,Pave,None,Reg,Lvl,AllPub,...,0.0,None,None,None,0.0,5.0,2007.0,WD,Normal,181500.0
2,3.0,60.0,RL,68.0,11250.0,Pave,None,IR1,Lvl,AllPub,...,0.0,None,None,None,0.0,9.0,2008.0,WD,Normal,223500.0
3,4.0,70.0,RL,60.0,9550.0,Pave,None,IR1,Lvl,AllPub,...,0.0,None,None,None,0.0,2.0,2006.0,WD,Abnorml,140000.0
4,5.0,60.0,RL,84.0,14260.0,Pave,None,IR1,Lvl,AllPub,...,0.0,None,None,None,0.0,12.0,2008.0,WD,Normal,250000.0


### Data preprocessing


In [114]:
# separate features and target
X = df_house.drop('SalePrice', axis=1)
y = df_house['SalePrice']

# fill missing values correctly
for col in X.columns:
    if str(X[col].dtype) == 'category' or X[col].dtype == 'object':
        X[col] = X[col].astype('object').fillna('missing')
    else:
        X[col] = X[col].fillna(X[col].median())

# encode categorical features
X = pd.get_dummies(X)

### Train XGBoost Regressor

In [115]:
from xgboost import XGBRegressor

# split data
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=0)

# create model
reg = XGBRegressor()

# train model
reg.fit(X_train, y_train)

# evaluate
print("train score (R^2)= {:.3f}".format(reg.score(X_train, y_train)))
print("test score (R^2)= {:.3f}".format(reg.score(X_test, y_test)))

train score (R^2)= 1.000
test score (R^2)= 0.843
